# Tapis tenant migration helper

This notebook migrates pods and volumes from `tacc.tapis.io` to `portals.tapis.io` using `tapipy`.


## 1. Install and Import Libraries

Load `tapipy`, shared helpers, and environment variables from `.env.example`.


In [ ]:
import os
import json
import getpass
from pathlib import Path
from dotenv import load_dotenv
from tapipy.tapis import Tapis

load_dotenv(Path('.env.example'))

def create_tapis_client(base_url, tenant_id, username, password):
    client = Tapis(base_url=base_url, tenant_id=tenant_id, username=username, password=password)
    client.get_tokens()
    return client

def tapis_to_jsonable(value):
    if isinstance(value, (str, int, float, bool)) or value is None:
        return value
    if isinstance(value, dict):
        return {k: tapis_to_jsonable(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [tapis_to_jsonable(v) for v in value]
    if hasattr(value, '__dict__'):
        return {
            k: tapis_to_jsonable(v)
            for k, v in vars(value).items()
            if not k.startswith('_') and not callable(v)
        }
    return str(value)

def save_json(path, data):
    Path(path).write_text(json.dumps(tapis_to_jsonable(data), indent=2))

def resource_items(response):
    if isinstance(response, list):
        return response
    if isinstance(response, dict):
        return response.get('result', response)
    result = getattr(response, 'result', None)
    return result if result is not None else response

def load_json_template(path):
    return json.loads(Path(path).read_text())

def replace_strings(value, replacements):
    if isinstance(value, dict):
        return {k: replace_strings(v, replacements) for k, v in value.items()}
    if isinstance(value, list):
        return [replace_strings(v, replacements) for v in value]
    if isinstance(value, str):
        result = value
        for old, new in replacements.items():
            result = result.replace(old, new)
        return result
    return value

print('Libraries imported successfully.')


## 2. Authenticate to Source Tenant

Create a source Tapis client for `tacc.tapis.io` using environment variables or interactive input.


In [ ]:
source_config = {
    'base_url': 'https://tacc.tapis.io',
    'tenant_id': 'tacc',
    'username': os.getenv('SOURCE_TAPIS_USERNAME', ''),
    'password': os.getenv('SOURCE_TAPIS_PASSWORD', ''),
}

if not source_config['username']:
    source_config['username'] = input('Source Tapis username: ')
if not source_config['password']:
    source_config['password'] = getpass.getpass('Source Tapis password: ')

source_client = create_tapis_client(**source_config)
print('Authenticated to source tenant:', source_config['base_url'])


## 3. List Source Pods and Volumes

Fetch source pods and volumes and save them locally for inspection.


In [ ]:
source_pods = source_client.pods.list_pods()
source_volumes = source_client.pods.list_volumes()
source_pod_items = resource_items(source_pods)
source_volume_items = resource_items(source_volumes)
print('Source pods count:', len(source_pod_items))
print('Source volumes count:', len(source_volume_items))
save_json('source_pods.json', source_pods)
save_json('source_volumes.json', source_volumes)
print('Saved source_pods.json and source_volumes.json')


## 4. Load Example Templates and Patch Tenant References

Patch local example JSON files from the source tenant hostnames to the destination tenant hostnames.


In [ ]:
template_paths = {
    'api': Path('api-example'),
    'ui': Path('exampleUpstream'),
    'postgres': Path('postgres_example'),
}

tenant_replacements = {
    'tacc.tapis.io': 'portals.tapis.io',
    'https://tacc.tapis.io': 'https://portals.tapis.io',
}

templates = {}
for name, template_path in template_paths.items():
    if template_path.exists():
        templates[name] = replace_strings(load_json_template(template_path), tenant_replacements)
        print(f'Loaded and patched template: {name}')
    else:
        print(f'Template file missing: {template_path}')

for name, template in templates.items():
    Path(f'patched_{name}_template.json').write_text(json.dumps(template, indent=2))
print('Wrote patched_*_template.json files')


## 5. Authenticate to Destination Tenant

Create a destination Tapis client for `portals.tapis.io`.


In [ ]:
dest_config = {
    'base_url': 'https://portals.tapis.io',
    'tenant_id': 'portals',
    'username': os.getenv('DEST_TAPIS_USERNAME', '') or os.getenv('SOURCE_TAPIS_USERNAME', ''),
    'password': os.getenv('DEST_TAPIS_PASSWORD', '') or os.getenv('SOURCE_TAPIS_PASSWORD', ''),
}

if not dest_config['username']:
    dest_config['username'] = input('Destination Tapis username: ')
if not dest_config['password']:
    dest_config['password'] = getpass.getpass('Destination Tapis password: ')

dest_client = create_tapis_client(**dest_config)
print('Authenticated to destination tenant:', dest_config['base_url'])


## 6. Create Matching Destination Pods and Volumes

Create missing destination resources while skipping any existing pod or volume IDs.


In [ ]:
def replace_resource_strings(resource):
    replacements = {
        'tacc.tapis.io': 'portals.tapis.io',
        'https://tacc.tapis.io': 'https://portals.tapis.io',
    }
    return replace_strings(resource, replacements)

def resource_payload_mapper(resource):
    payload = replace_resource_strings(tapis_to_jsonable(resource))
    payload = dict(payload)
    for field in [
        'id', 'created', 'updated', 'creation_ts', 'update_ts', 'status',
        'status_container', 'status_requested', 'tenant', 'site', 'k8_name',
        'logs', 'permissions'
    ]:
        payload.pop(field, None)
    for networking_obj in (payload.get('networking', {}) or {}).values():
        if isinstance(networking_obj, dict):
            networking_obj.pop('cors_allow_origins', None)
    if 'tenant_id' in payload:
        payload['tenant_id'] = dest_config['tenant_id']
    return payload

def existing_resource_ids(resources, id_field):
    ids = set()
    for resource in resource_items(resources):
        resource_data = tapis_to_jsonable(resource)
        resource_id = resource_data.get(id_field)
        if resource_id:
            ids.add(resource_id)
    return ids

def create_destination_resources(resources, existing_ids, create_fn, id_field, label):
    created = []
    skipped = []
    for resource in resources:
        payload = resource_payload_mapper(resource)
        resource_id = payload.get(id_field, '<unknown>')
        if resource_id in existing_ids:
            skipped.append(resource_id)
            print(f'Skipping existing {label}: {resource_id}')
            continue
        try:
            result = create_fn(**payload)
            created.append(tapis_to_jsonable(result))
            existing_ids.add(resource_id)
            print(f'Created {label}: {resource_id}')
        except Exception as exc:
            print(f'Failed to create {label} {resource_id}: {exc}')
    return created, skipped

dest_existing_volume_ids = existing_resource_ids(dest_client.pods.list_volumes(), 'volume_id')
dest_existing_pod_ids = existing_resource_ids(dest_client.pods.list_pods(), 'pod_id')

created_volumes, skipped_volumes = create_destination_resources(
    source_volume_items,
    dest_existing_volume_ids,
    dest_client.pods.create_volume,
    'volume_id',
    'volume',
)
created_pods, skipped_pods = create_destination_resources(
    source_pod_items,
    dest_existing_pod_ids,
    dest_client.pods.create_pod,
    'pod_id',
    'pod',
)
save_json('created_destination_volumes.json', created_volumes)
save_json('created_destination_pods.json', created_pods)
save_json('skipped_destination_volumes.json', skipped_volumes)
save_json('skipped_destination_pods.json', skipped_pods)
print('Saved created_destination_volumes.json, created_destination_pods.json, skipped_destination_volumes.json, and skipped_destination_pods.json')


## 7. Verify Tenant Resource Migration

Verify destination pods and volumes after the tenant resource migration.


In [ ]:
dest_pods = dest_client.pods.list_pods()
dest_volumes = dest_client.pods.list_volumes()
print('Destination pods count:', len(resource_items(dest_pods)))
print('Destination volumes count:', len(resource_items(dest_volumes)))
